# 01 — LLM Bootstrap Labelling

Labels the **LLM Bootstrap** partition with Gemini to seed the HITL classifier training set.
Replaces — or precedes — the human seed step in `classification_strategy.md` (Step 0).

**Label set:** `originality` / `none`. Definitions, decision test, worked examples and
exclusions all live in `categories.md`, which is the source of truth; the criteria are pasted
into the Categories cell below because Colab does not clone this repo at runtime.

**Inputs:** `llm_bootstrap_dataset.pkl` (carved out by `00_hitl_data_preparation.ipynb`,
disjoint from the Base / HITL / Inference partitions — see `partition_ids.pkl`).

**Outputs:** `llm_bootstrap_labels.csv` (HITL schema), `llm_bootstrap_labels_full.pkl`
(+ `confidence`, `rationale`), a per-run usage record, and the seen-ids basket.

**Spend guards:** `MAX_LLM_TWEETS` rows and `MAX_SESSION_TOKENS` tokens per run, both binding
regardless of `SMOKE_TEST`. Run the **Preflight** section before the labelling loop — it costs
one call and catches a broken request shape that would otherwise take a full run to surface.

---
*Sections below are collapsed by default. Click a heading to expand it.*

# 1 · Setup

Environment switch, dependencies, imports, all tunable parameters, the label taxonomy, the
API client, and the per-tweet classification function. Nothing here calls the API.

## Environment and Paths

In [ ]:
%%time
import os
from pathlib import Path
import sys

# --- ENVIRONMENT SWITCH ---
# True  → local machine with Google Drive Desktop mounted
# False → Google Colab cloud
RUNNING_LOCALLY = False

# --- DATASET TYPE ---
# 'AI'  → AItrust_twits_pruned_dict.json    → Partitioned Data/AI Data/
# 'Art' → AItrust_Art_pruned_twit_dict.json → Partitioned Data/Art Data/
DATASET_TYPE = 'AI'

if RUNNING_LOCALLY:
    _REPO_ROOT = str(Path(os.getcwd()).resolve().parents[1])
    if _REPO_ROOT not in sys.path:
        sys.path.insert(0, _REPO_ROOT)
    BASE_PATH = Path('/Volumes/GoogleDrive/My Drive/Colab Projects/AI Public Trust')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/My Drive/Colab Projects/AI Public Trust')

# Pre-compute critical paths
twits_folder          = BASE_PATH / 'Raw Data/Twits/'
test_folder           = BASE_PATH / 'Raw Data/'
datasets_folder       = BASE_PATH / 'Data Sets'
cleanedds_folder      = BASE_PATH / 'Data Sets/Cleaned Data'
networks_folder       = BASE_PATH / 'Data Sets/Networks/'
literature_folder     = BASE_PATH / 'Literature/'
topic_models_folder   = BASE_PATH / 'Models/Topic Modeling/'
classifiers_folder    = BASE_PATH / 'Models/Classifiers/'
classifiers_folder.mkdir(parents=True, exist_ok=True)
partitioned_folder    = cleanedds_folder / 'Partitioned Data' / f'{DATASET_TYPE} Data'
hitl_folder = datasets_folder / 'Classifiers_Data' / 'HITL'

## Dependencies

In [ ]:
%%time
import os
if not RUNNING_LOCALLY:
    print('Running Colab setup...')
    import subprocess
    subprocess.run(['pip', 'install', '-q', 'google-genai'])
else:
    print('Running locally: skipping Colab setup.')

## Imports

In [ ]:
%%time
import hashlib
import json
import re
import time
import textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import tqdm
from google import genai
from google.genai import types

## Configuration

Every tunable knob for this notebook. `SMOKE_TEST` scales the run down; `MAX_LLM_TWEETS` and
`MAX_SESSION_TOKENS` are hard ceilings that bind either way.

In [ ]:
%%time
# ── Run mode ────────────────────────────────────────────────────────────
SMOKE_TEST   = True       # True → label SMOKE_TEST_N tweets only; flip to False for the full run
SMOKE_TEST_N = 500

# True unassigns the Colab runtime in the final cell, freeing the instance once the run
# is done. It also DESTROYS every variable, including out_df — so the Review section
# (section 5) can no longer be re-run interactively afterwards. That is survivable
# because Review executes BEFORE teardown on a Run All, so its report is already printed
# and saved in the notebook output; and the same data is on disk in
# llm_bootstrap_labels_full.pkl, which Review could be re-pointed at.
# Set back to False when iterating on categories.md and re-reading rows by hand.
# (NOTEBOOK_WRITING_SKILL.md §11: gate the disconnect behind a parameter.)
AUTO_DISCONNECT = True

# Absolute ceiling on how many tweets are ever sent to the LLM in a single run.
# Enforced at three independent layers (dataframe truncation in Load Input, an
# assertion before the loop, and a per-call counter inside classify_tweet), so it
# binds even when SMOKE_TEST is False. The prompt in llm_bootstrap_prompt.md is
# still being tuned - raise this deliberately, never as a side effect of flipping
# SMOKE_TEST to False. Note it caps TWEETS, not calls: with N_LABEL_PASSES passes
# the request ceiling is the product (2,000 x 2 = 4,000).
MAX_LLM_TWEETS = 2_000

# Independent labelling passes over the SAME tweets. 1 = single pass (no agreement data).
# 2 gives test-retest reliability: an LLM label is a measurement, and without a repeat
# there is no error bar on it. Disagreements between passes are the highest-value rows
# for human review — they mark where the criteria under-determine the answer.
# COST AND CALLS SCALE LINEARLY: N passes over N_tweets issues N x N_tweets requests.
N_LABEL_PASSES = 2

# Per-pass temperature. None -> TEMPERATURE for every pass.
#   [0.0, 0.0] measures DETERMINISM (agreement should be near-total; disagreement means
#              the model is unstable even at temperature 0).
#   [0.0, 0.7] measures ROBUSTNESS to sampling — a harsher and more informative test of
#              whether the criteria pin the answer down.
#   [0.0, 0.3] the middle setting, and the one in use: pass 1 stays the deterministic
#              operative label, pass 2 samples just enough to shake loose the rows where
#              the criteria leave real room. Changes what a disagreement MEANS — it is no
#              longer evidence of model instability but of an underdetermined rule, which
#              is the more useful signal while the fence is being tuned. Expect agreement
#              to fall below the near-100% a [0.0, 0.0] run gives; that drop is the
#              measurement, not a regression.
PASS_TEMPERATURES = [0.0, 0.3]

# Seed for the fixed permutation the run selects from (see Load Input). Changing it
# re-draws WHICH tweets get labelled, so a previously-labelled set is no longer a
# prefix of the new one. Treat it as frozen once the first real run has happened.
SELECTION_SEED = 42

# Hard ceiling on total tokens (input + output) spent in a single run of this notebook.
# Checked BETWEEN calls — usage_metadata only arrives with a response — so the run stops
# at the first call after the ceiling is crossed; overshoot is bounded by one call.
#
# This is a BACKSTOP, not the binding limit, and it has to be sized against the CALL
# ceiling (MAX_LLM_TWEETS x N_LABEL_PASSES = 4,000), not against the tweet count: at
# ~1,400 tokens per call a run that actually reaches the cap costs ~5.6M. Sized below
# that, this fires first and silently truncates the run — at 2M it would have stopped a
# 1,000-tweet two-pass run (~2.2M measured) partway through pass 2, and the completeness
# filter in the Run cell would then have discarded every tweet pass 2 never reached.
# The row cap is meant to be what stops a run; this only fires if something unexpected
# inflates per-call cost — a much longer prompt, or a model whose thinking budget is not
# actually off. If it does fire, the run stops cleanly: completed rows are saved and
# basketed as normal.
MAX_SESSION_TOKENS = 6_000_000

# ── LLM ─────────────────────────────────────────────────────────────────
# Gemini model id. Verify against the Available Models cell below, which lists what this
# key can actually reach — model availability shifts and a retired id 404s every call.
# Measured against this project's key on 2026-08-03 with the real prompt:
#   'gemini-3.1-flash-lite' — DEFAULT. Only candidate that ACCEPTS thinking_budget=0, and
#                             parses reliably (1,028 in / 50-65 out per tweet).
#   'gemini-3.5-flash-lite' — works, but REJECTS thinking_budget=0 (400 INVALID_ARGUMENT),
#                             so thinking cost is uncontrolled.
#   'gemini-3.6-flash'      — rejects thinking_budget=0 AND spent 118 thinking tokens, then
#                             truncated the answer at MAX_OUTPUT_TOKENS -> UNPARSEABLE.
#   'gemini-2.5-*'          — RETIRED for new users: 404 on every call, even though
#                             models.list() still advertises them. Do not use.
# See content/how-to/LLM_ERROR_HANDLING_SKILL.md.
MODEL_NAME  = 'gemini-3.1-flash-lite'
TEMPERATURE = 0.0         # deterministic classification; raise only if you want sampling diversity

# Per-pass model. None -> MODEL_NAME for every pass.
#
# Two temperatures of ONE model share a prior, so their agreement partly measures the
# model's self-consistency. Two DIFFERENT models agreeing is the stronger test: it says
# the criteria survive a change of prior, which is what you actually want to know about
# the fence in llm_bootstrap_prompt.md. A disagreement between models points at the
# criteria; a disagreement between temperatures can just be sampling noise.
#
#   None                                            -> MODEL_NAME for every pass
#   ['gemini-3.1-flash-lite', 'gemini-3.5-flash-lite'] -> cross-model agreement
#
# Every DISTINCT model here is probed for thinking-budget support and preflighted with the
# real prompt before the loop starts, because they disagree about what they accept:
# 3.1-flash-lite takes thinking_budget=0, 3.5-flash-lite and 3.6-flash reject it with a
# 400. Without the per-model probe, a config valid for pass 1 fails on every call of pass 2.
PASS_MODELS = None

# Resolved once here so every cell below agrees on which models are in play.
PASS_MODELS_RESOLVED = list(PASS_MODELS or [MODEL_NAME] * N_LABEL_PASSES)
assert len(PASS_MODELS_RESOLVED) >= N_LABEL_PASSES, (
    f'PASS_MODELS has {len(PASS_MODELS_RESOLVED)} entries but N_LABEL_PASSES is {N_LABEL_PASSES}')
PASS_MODELS_RESOLVED = PASS_MODELS_RESOLVED[:N_LABEL_PASSES]
ACTIVE_MODELS = list(dict.fromkeys(PASS_MODELS_RESOLVED))   # distinct, order preserved

# TOKENOPT_REF.md §4: always cap output tokens; 64-256 is the band for classification.
# 128 covers {label, confidence, one-sentence rationale}. If rationales come back
# truncated (they surface as PARSE_ERROR rows), raise this before anything else.
MAX_OUTPUT_TOKENS = 128

# ── Token accounting (TOKENOPT_REF.md §16-17) ───────────────────────────
# Dollars per 1M tokens, keyed by model so switching MODEL_NAME cannot leave a stale rate
# behind. VERIFY against https://ai.google.dev/pricing before trusting a cost figure —
# provider rates change and these go out of date silently. Reporting only; they do not
# gate the run, so a wrong rate misleads but never overspends.
# UNVERIFIED for the 3.x family — placeholders at the historical flash-lite tier. Check
# https://ai.google.dev/pricing and correct these before quoting any cost figure.
MODEL_RATES = {
    'gemini-3.1-flash-lite': (0.10, 0.40),   # PLACEHOLDER — verify
    'gemini-3.5-flash-lite': (0.10, 0.40),   # PLACEHOLDER — verify
    'gemini-2.5-flash':      (0.30, 2.50),
    'gemini-2.5-flash-lite': (0.10, 0.40),
}
FALLBACK_RATES = (0.30, 2.50)
_unpriced = [_m for _m in ACTIVE_MODELS if _m not in MODEL_RATES]
if _unpriced:
    print(f'No rates recorded for {_unpriced} — cost figures will use the fallback rate '
          f'{FALLBACK_RATES} and be WRONG. Add them to MODEL_RATES.')

# Cost is accounted at the MOST EXPENSIVE rate in use — the highest input rate and the
# highest output rate across ACTIVE_MODELS, taken independently so the bound holds
# whatever the token mix turns out to be. With mixed models the reported figure is an
# UPPER BOUND on spend rather than the actual bill; that is the point, since a cost
# number that can under-report is worse than one that is merely pessimistic. With a
# single model (the default) it is exact.
COST_PER_M_INPUT  = max(MODEL_RATES.get(_m, FALLBACK_RATES)[0] for _m in ACTIVE_MODELS)
COST_PER_M_OUTPUT = max(MODEL_RATES.get(_m, FALLBACK_RATES)[1] for _m in ACTIVE_MODELS)
RATES_ARE_WORST_CASE = len(ACTIVE_MODELS) > 1
if RATES_ARE_WORST_CASE:
    print(f'{len(ACTIVE_MODELS)} models in use {ACTIVE_MODELS} — costing everything at the '
          f'dearest rate (${COST_PER_M_INPUT}/M in, ${COST_PER_M_OUTPUT}/M out): '
          f'reported spend is an upper bound.')
COST_ALERT_USD    = 1.00   # pre-flight prints a loud warning above this
CHARS_PER_TOKEN   = 3.5    # English average, used only for the pre-flight estimate

# Thinking budget. 0 disables the internal reasoning chain, which for one-shot
# classification is pure cost — and worse than cost: gemini-3.6-flash spent 118 thinking
# tokens and then TRUNCATED its answer at MAX_OUTPUT_TOKENS, returning unparseable text.
# Not every model accepts 0 (3.5-flash-lite and 3.6-flash reject it with 400
# INVALID_ARGUMENT); the API Key cell probes once and falls back to the model default if
# so, printing a warning. Set to a positive integer to bound rather than disable thinking.
THINKING_BUDGET = 0

# Strict server-side response schema (TOKENOPT_REF.md §5). When True the API enforces the
# label enum and cannot return prose or fenced JSON. Set False to fall back to asking for
# JSON in the prompt and stripping fences client-side — the notebook's original behaviour.
#
# Verified working on gemini-3.1-flash-lite with the real prompt (2026-08-03). The earlier
# 100% failure rate was model retirement and quota exhaustion, not the schema.
USE_RESPONSE_SCHEMA = True

# ── Retry / backoff / rate limiting ─────────────────────────────────────
MAX_RETRIES     = 3
INITIAL_BACKOFF = 2.0     # seconds; doubled on each retry

# A 429 RESOURCE_EXHAUSTED is the one failure retrying cannot fix — it spends more of
# exactly what you have run out of. With MAX_RETRIES=3 a 100-row run issues 300
# requests, which is how a free-tier quota gets burned in a single failed run.
# On a 429 the notebook waits these many seconds and retries, in order; if the last one
# still 429s, the run aborts (completed rows are saved). Long waits let an RPM window
# reset — LLM_ERROR_HANDLING_SKILL.md notes 5s/10s is usually too short. Set to [] to
# abort on the first quota error, which is the right setting on a free tier where a 429
# means genuine exhaustion rather than throttling.
QUOTA_BACKOFF_S = [15, 30]

# Total 429s tolerated across the whole run before aborting regardless of the backoff
# above. Stops a slow bleed where every row costs two long waits before succeeding.
MAX_QUOTA_ERRORS = 5

# Minimum seconds between requests, to stay under the tier's requests-per-minute cap.
# PAID tier: 0.0 — limits are far above what a serial loop can reach.
# FREE tier: ~4.0 (gemini-2.5-flash-lite free limits are on the order of 15 RPM).
# Check actual limits at https://ai.dev/rate-limit; per LLM_ERROR_HANDLING_SKILL.md the
# quota page for AI Studio keys is Cloud Console → the key's project → APIs & Services
# → Generative Language API → Quotas.
REQUEST_INTERVAL_S = 0.0

# ── I/O ─────────────────────────────────────────────────────────────────
INPUT_PATH        = partitioned_folder / 'llm_bootstrap_dataset.pkl'
OUTPUT_CSV        = hitl_folder / 'llm_bootstrap_labels.csv'
OUTPUT_PKL        = hitl_folder / 'llm_bootstrap_labels_full.pkl'
CHECKPOINT_PREFIX = 'llm_bootstrap_checkpoint'
CHECKPOINT_EVERY  = 1_000  # save partial results every N tweets

## Label Set

The live taxonomy is **two labels**: `originality` (the tweet appeals to newness, creativity,
copying or theft **as a criterion for the value of art**) and `none` (the residual bucket).

This cell defines only the *list*, which is what the response schema's `enum` is built from.
The criteria the model actually reads live in `llm_bootstrap_prompt.md` beside this notebook
and are embedded by the Prompt Builder cell below. `categories.md` explains why the taxonomy
is shaped this way; it no longer carries a second copy of the prompt text.

In [ ]:
%%time
# Closed label set. `none` is the residual bucket, not the negation of `originality` — it
# keeps its meaning when further categories are added.
#
# This list drives the enum in the response schema sent to the API. The prompt states the
# label set in its own words, in llm_bootstrap_prompt.md; the Prompt Builder cell asserts
# the two agree, so adding a category here and not there fails loudly instead of producing
# a model that confidently returns a label the schema rejects.
CATEGORIES: list[str] = [
    'intentionalism',
    'anti_intentionalism',
    'cognitivism',
    'expressivism',
    'hedonism',
    'originality',
    'achievement',
    'none',
]

assert CATEGORIES, 'CATEGORIES is empty — fill it in before running.'
print(f'{len(CATEGORIES)} categories: {", ".join(CATEGORIES)}')

## API Key and Client

On Colab the key comes from **Colab Secrets** — sidebar → 🔑 → name it `GEMINI_API_KEY` and
toggle **Notebook access** on for this notebook. Locally it comes from the `GEMINI_API_KEY`
environment variable. The key is never written into this file.

In [ ]:
%%time
API_KEY = ''  # leave empty — populated below from Colab secrets / env var

if not RUNNING_LOCALLY:
    try:
        from google.colab import userdata
        API_KEY = userdata.get('GEMINI_API_KEY')
    except Exception as e:
        print(f'Colab userdata lookup failed: {e}')
else:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, 'GEMINI_API_KEY not set. Add it as a Colab secret or env var before running.'
assert MODEL_NAME, 'MODEL_NAME is empty — set it in the Configuration cell.'
assert all(ACTIVE_MODELS), 'PASS_MODELS contains an empty model name.'

client = genai.Client(api_key=API_KEY)

config_kwargs = dict(
    temperature=TEMPERATURE,
    max_output_tokens=MAX_OUTPUT_TOKENS,   # TOKENOPT_REF.md §4
    response_mime_type='application/json',
)

# TOKENOPT_REF.md §5 — native structured output. The schema is enforced server-side, so
# the model cannot return a label outside CATEGORIES and cannot wrap its JSON in prose or
# markdown fences (the two things that produced PARSE_ERROR rows). The enum is derived
# from CATEGORIES, so adding a category needs no edit in this cell.
RESPONSE_SCHEMA = {
    'type': 'array',
    'items': {
        'type': 'object',
        'properties': {
            'category':   {'type': 'string', 'enum': CATEGORIES},
            'confidence': {'type': 'number'},
            'rationale':  {'type': 'string'},
        },
        'required': ['category', 'confidence', 'rationale'],
    }
}
if USE_RESPONSE_SCHEMA:
    if 'response_json_schema' in types.GenerateContentConfig.model_fields:
        config_kwargs['response_json_schema'] = RESPONSE_SCHEMA
    else:
        # older google-genai releases name the field response_schema
        config_kwargs['response_schema'] = RESPONSE_SCHEMA
    print('Strict response schema ENABLED — run the Preflight cell before the loop.')
else:
    print('Strict response schema disabled; falling back to prompt-instructed JSON '
          '+ client-side fence stripping.')
# Thinking budget, probed rather than assumed. Model families disagree about whether
# thinking_budget can be set at all, and guessing wrong costs a whole run: the failure is
# a 400 on every call. One 2-token probe settles it here instead.
def _probe(kwargs, model) -> str:
    """Return '' if the config is accepted by that model, else the error text."""
    try:
        client.models.generate_content(
            model=model, contents='ping',
            config=types.GenerateContentConfig(**{**kwargs, 'max_output_tokens': 8}))
        return ''
    except Exception as e:
        return f'{type(e).__name__}: {str(e)[:110]}'

# Probed PER MODEL, because models disagree about whether thinking_budget can be set at
# all. One shared config would be valid for pass 1's model and 400 on every call of
# pass 2's. MODEL_KWARGS holds the accepted config for each distinct model; the run loop
# layers that pass's temperature on top.
MODEL_KWARGS: dict = {}
for _m in ACTIVE_MODELS:
    _wanted = {**config_kwargs,
               'thinking_config': types.ThinkingConfig(thinking_budget=THINKING_BUDGET)}
    _err = _probe(_wanted, _m)
    if not _err:
        MODEL_KWARGS[_m] = _wanted
        print(f'  {_m}: thinking budget pinned to {THINKING_BUDGET}')
    else:
        print(f'  WARNING: {_m} rejected thinking_budget={THINKING_BUDGET} — {_err}')
        print('    Falling back to the model default: thinking tokens are UNBOUNDED and can')
        print('    consume MAX_OUTPUT_TOKENS before the answer, producing PARSE_ERROR rows.')
        print('    Watch the measured token report, or use gemini-3.1-flash-lite.')
        _err2 = _probe(config_kwargs, _m)
        assert not _err2, f'{_m} rejected the base config too: {_err2}'
        MODEL_KWARGS[_m] = dict(config_kwargs)

# Default config, used by any cell that does not name a pass (preflight iterates itself).
GEN_CONFIG = types.GenerateContentConfig(**MODEL_KWARGS[ACTIVE_MODELS[0]])
print(f'LLM client ready. Pass models: {PASS_MODELS_RESOLVED}')

## Available Models

Validates `MODEL_NAME` against what this key can actually reach. Quiet when the model is
available; when it is not, it says so and lists the Gemini text models that are — which is
the difference between a two-second fix and a run where every call 404s.

Model availability shifts: `gemini-2.5-flash-lite` was retired for new users on 2026-08-03
and had been this notebook's default.

In [ ]:
%%time
_available = sorted(
    m.name.replace('models/', '')
    for m in client.models.list()
    if not getattr(m, 'supported_actions', None) or 'generateContent' in m.supported_actions
)
_text_models = [
    _n for _n in _available
    if _n.startswith('gemini') and not any(
        _s in _n for _s in ('embedding', 'tts', 'image', 'live', 'native-audio', 'computer-use'))
]
print(f'{len(_available)} models support generateContent; {len(_text_models)} are gemini text models:')
for _n in _text_models:
    print('   ', _n, '   <-- in PASS_MODELS' if _n in ACTIVE_MODELS else '')

# Listing is NECESSARY BUT NOT SUFFICIENT. Observed 2026-08-03: models.list() reported
# gemini-2.5-flash while generate_content on it returned 404 "no longer available to new
# users" — the retirement is enforced at call time, not in the catalogue. So this cell
# cannot confirm a model works; only the Preflight call can. It still catches the clear
# case of a name that is not offered at all.
for _m in ACTIVE_MODELS:
    if _m not in _available:
        print(f"\nWARNING: '{_m}' is not in the catalogue at all — expect a 404.")
    else:
        print(f"\n'{_m}' is listed — but listing does NOT prove it is callable "
              f"(gemini-2.5-flash is listed here and 404s). Preflight is the real check.")

## Prompt Builder

`llm_bootstrap_prompt.md` — the single authored copy of the prompt, sitting beside this
notebook in the repo — embedded verbatim, then split on its fence to recover the exact text
that is sent. Static content first, tweet last, so the fixed prefix stays implicit-cache
eligible.

**The literal below is generated. Do not edit it here.** Edit `llm_bootstrap_prompt.md` and
run `python3 notebooks/05_Classifiers/sync_prompt.py`; `--check` fails while the two differ.

In [ ]:
%%time
# The prompt lives in llm_bootstrap_prompt.md, beside this notebook in the repo. It is
# embedded here verbatim because Colab does not clone the repo at runtime, and it is
# written back out unchanged in the Save section, so the copy that lands next to the
# labels is byte-identical to the repo's.
#
# DO NOT EDIT THE LITERAL BY HAND. Edit llm_bootstrap_prompt.md, then run:
#     python3 notebooks/05_Classifiers/sync_prompt.py
# Hand edits here are silently reverted by the next sync, and `sync_prompt.py --check`
# fails while they are present.
# ── BEGIN GENERATED — llm_bootstrap_prompt.md ──────────────────────────
PROMPT_DOC: str = '''\
# LLM Bootstrap Labelling — Prompt

**This file is the prompt.** The text inside the fence below is sent to the model verbatim,
once per tweet, with `{{TWEET}}` replaced by the tweet being classified and nothing else
added. There is no system instruction, no conversation history, and no context carried
between calls: whatever the model knows about this task, it knows from the fence below.

It is the single authored copy. `notebooks/05_Classifiers/01_llm_bootstrap_labelling.ipynb`
embeds this file verbatim as `PROMPT_DOC` (Colab does not clone the repo at runtime, so it
cannot be read from disk there) and writes it back out unchanged next to the labels it
produces. The copy sitting beside `llm_bootstrap_labels.csv` in Google Drive is byte-identical
to this one — that is asserted, not hoped for.

`categories.md` in this folder explains *why* the taxonomy is shaped this way — what `none`
means, the expected class balance, what has to be decided before adding a category. This file
is what the model is actually told.

## Editing this file

Edit the fence, then run the sync script, then re-run the notebook:

```bash
python3 notebooks/05_Classifiers/sync_prompt.py
```

The script pastes this file into the notebook's `c-prompt` cell and verifies the round-trip.
**Editing the notebook cell by hand instead will be overwritten**, and the sync script fails
loudly if the two have diverged, so the repo can never disagree with itself about what was
sent. Nothing outside the fence reaches the model — the prose in this file is for you and for
whoever reviews the labels.

The tuning loop that this is built for: run with `SMOKE_TEST = True`, open
`llm_bootstrap_labels_full.pkl`, sort by `confidence` ascending and read the bottom rows —
those are where the criteria are underspecified. Edit the fence, sync, re-run.

## The prompt

````text
You are a tweet classifier for a research project on AI public trust.
This is a multi-label classification task. Evaluate the tweet against the following conceptual frameworks used to answer the questions "What is art?" and "What makes art valuable?".

Classify the tweet into ALL of the following categories that apply:
intentionalism, anti_intentionalism, cognitivism, expressivism, hedonism, originality, achievement, none

Per-category criteria:

### intentionalism
Definition: This framework posits that the artist's intent is the ultimate source of meaning and the defining characteristic of art. A work is art because it was intended to be so, and it means exactly what the creator meant it to mean. Tweets using this framework will often argue that AI cannot produce art because it lacks a conscious mind, meaning, or deliberate intention.
Example: "can we please stop calling it 'ai art' and start calling it 'ai generated images' instead? art requires meaning and intention, ai has none of that"

### anti_intentionalism
Definition: In direct contrast to intentionalism, this view argues that the author's intent does not govern the meaning of the work. Instead, meaning is derived from the work itself (the language, imagery, or form) and the audience's interpretation. Tweets in this category might invoke the "death of the author" or emphasize that the image speaks for itself regardless of who or what made it.

### cognitivism
Definition: This framework values art for its ability to impart knowledge, foster understanding, or provoke thought. Art is seen as a vehicle for discovering or communicating ideas. Tweets applying this framework will praise art that teaches us something or makes us think, or conversely criticize art for being shallow or intellectually empty.
Example: "gm ☕ i've been playing a lot with ai image generation lately... i like to think i'm a creative person, so for me it's an avenue to express those ideas and a journey of discovery.☄️"

### expressivism
Definition: Expressivism grounds the value (and often the definition) of art in emotion. This can refer to the emotion the artist felt during creation, the emotion captured within the piece, or the feelings it evokes in the audience. Tweets in this category will evaluate works based on their emotional resonance, or dismiss AI art by pointing out its lack of "feeling" or inability to experience emotion.
Example: "🖤 creatures - this ai collection is more about feeling than about a fairy tale🖤"

### hedonism
Definition: This view asserts that the primary value of art lies in the pleasure it brings the viewer. Aesthetic hedonism evaluates art based on whether it is enjoyable, pleasant, or makes the audience feel good. Negative evaluations under this framework will dismiss art simply because it is ugly or unpleasant to look at.

### originality
Definition: Originality values the novel, the creative, and the unprecedented. To be good art (or art at all), the work must offer something non-trivially new. In discourse around AI, this framework is frequently invoked in the negative—criticizing works for being derivative, stolen, traced, copied, or plagiarized from real artists.
Examples:
- "i don't mind ai art as long as it's not plagiarism..."
- "ai is just fancy photoshop for thieves."

### achievement
Definition: This framework ties the value and definition of art to the human effort, skill, and mastery required to produce it. Art is viewed as a triumph of hard work. Tweets using this framework will often attack AI art for being a shortcut, requiring no real skill or patience, and bypassing the arduous learning process that makes traditional art valuable.
Example: "This AI collaboration art thing, isn't effort."

### none
Definition: No category above applies. This is the residual bucket — it is not a claim that the tweet is unrelated to art or to AI. Only output `none` if NO other category applies.

Confidence.
0.8-1.0   Explicit: the vocabulary is present and the link to art's worth/definition is stated outright.
0.5-0.79  Implicit: the appeal is inferred from framing, or shares the tweet with a competing theme.
0.0-0.49  Contested: one plausible reading supports the label, another equally plausible reading does not.
Score confidence for whichever label(s) you chose.

In `rationale`, quote the phrase from the tweet that decided the label.

Return ONLY a JSON array of objects with this exact schema (no prose, no markdown fences). Each object in the array represents a selected category:
[
  {"category": "<one of the categories above>", "confidence": <number between 0 and 1>, "rationale": "<one short sentence>"}
]

Tweet:
"""{{TWEET}}"""
````

## What constrains the reply

The reply shape is enforced **server-side** by the API through a JSON schema, not merely
requested in the prose above. `category` cannot come back as anything outside the label set, and
the reply cannot be wrapped in prose or markdown fences.

```json
{
  "type": "array",
  "items": {
    "type": "object",
    "properties": {
      "category":   {"type": "string", "enum": ["intentionalism", "anti_intentionalism", "cognitivism", "expressivism", "hedonism", "originality", "achievement", "none"]},
      "confidence": {"type": "number"},
      "rationale":  {"type": "string"}
    },
    "required": ["category", "confidence", "rationale"]
  }
}
```

The enum is generated from the notebook's `CATEGORIES` list rather than copied from here, and
the notebook asserts that list against the category line in the fence above. Adding a category
therefore means editing three things that are checked against each other: `CATEGORIES`, the
category line in the fence, and the criteria describing it.

The notebook also caps output at 512 tokens and pins the thinking budget to 0 where the model
accepts it. If rationales start coming back truncated they surface as `PARSE_ERROR` rows;
raise `MAX_OUTPUT_TOKENS` before suspecting anything else.

## The CSV this produces

`llm_bootstrap_labels.csv`, one row per tweet, in the same schema as a human
`hitl_review_batch_*.csv`. Since it's multi-label, the CSV contains one-hot columns (0 or 1) for each category.

| Column | Meaning |
| :--- | :--- |
| `id` | Tweet id. |
| `text` | The tweet verbatim. This exact string is what replaced `{{TWEET}}`. |
| `likes`, `retweets` | Engagement counts, for your context only. **Not** part of the prompt. |
| `pred_intentionalism`, etc | The model's binary one-hot prediction for each category (1 if selected, 0 if not). |
| `human_intentionalism`, etc | **Yours.** Empty on delivery — fill in 0 or 1 for every row you review. |

The model's `confidence`, its `rationale`, and whether the independent passes agreed are
**not** in the CSV. They are in the sibling `llm_bootstrap_labels_full.pkl`.

Before overriding a label, read the fence. A label that looks wrong is often the criteria
working exactly as written — which is a reason to edit this file, not just that row.

## How predictions are decided

- The same tweet is labelled **more than once**, independently, each pass through a fresh client.
- **Pass 1 is the operative label** — the one in the CSV. Where passes disagree the row is flagged in `passes_agree` (in the pickle).
- `confidence` in the pickle is the **mean across passes**. It is the model's own self-report.
'''
PROMPT_DOC_SHA256 = '38c352174afb'   # of llm_bootstrap_prompt.md at sync time
# ── END GENERATED ─────────────────────────────────────────────────────

# Everything the model receives is inside the single ```` fence; the prose around it is
# for humans and never leaves this notebook. Splitting on the fence rather than on a
# line number means reordering the document cannot silently change the prompt.
_parts = PROMPT_DOC.split('````')
assert len(_parts) == 3, (
    f'expected exactly one four-backtick fence in llm_bootstrap_prompt.md, '
    f'found {max(len(_parts) - 1, 0) // 2}')
PROMPT_TEMPLATE: str = _parts[1].split('\n', 1)[1].rstrip('\n')   # drop the 'text' info string

# The tweet is the only variable part of the prompt, and it goes last so that everything
# before it is byte-identical on every call and stays eligible for the implicit cache.
assert PROMPT_TEMPLATE.count('{{TWEET}}') == 1, (
    'llm_bootstrap_prompt.md must contain exactly one {{TWEET}} placeholder inside the fence')

# CATEGORIES drives the response-schema enum in code; the fence states the label set in
# prose. Neither is derived from the other, so they are checked against each other here —
# a category added to one and not the other is the drift that would otherwise show up as
# the model confidently using a label the schema rejects.
assert ', '.join(CATEGORIES) in PROMPT_TEMPLATE, (
    f'the prompt does not list the label set as CATEGORIES has it: {", ".join(CATEGORIES)!r}')


def build_prompt(tweet_text: str) -> str:
    return PROMPT_TEMPLATE.replace('{{TWEET}}', tweet_text)


print(f'prompt: {len(PROMPT_TEMPLATE):,} chars '
      f'(~{len(PROMPT_TEMPLATE) // 4:,} tokens, re-sent on every call), '
      f'sha256:{PROMPT_DOC_SHA256}')

## Classification Function

One tweet → prompt → API call → parsed JSON, with retries, both spend guards, and token
accounting. The first failure of a run prints immediately rather than hiding behind retries.

In [ ]:
%%time
PARSE_ERROR_RESULT = [{'category': 'PARSE_ERROR', 'confidence': 0.0, 'rationale': ''}]

# Innermost layer of the MAX_LLM_TWEETS cap. classify_tweet refuses to issue a request
# once the budget is spent, so the ceiling holds even if the dataframe is sliced
# elsewhere or this loop is re-entered by hand. The Run Classification cell resets the
# counter before it starts, which keeps Restart-and-Run-All clean.
_llm_calls_made = 0
_first_error_reported = False


# TOKENOPT_REF.md §16 — accumulate the counts the API actually bills, as opposed to
# the pre-flight estimate. Recorded per request, including retries and responses that
# fail to parse: those are billed too, so leaving them out understates the real spend.
_usage = {'calls': 0, 'input': 0, 'output': 0, 'cached': 0, 'total': 0}


def _reset_usage() -> None:
    _usage.update(calls=0, input=0, output=0, cached=0, total=0)


def _record_usage(resp) -> None:
    meta = getattr(resp, 'usage_metadata', None)
    if meta is None:            # some SDK versions / error paths omit it
        return
    _usage['calls']  += 1
    _usage['input']  += getattr(meta, 'prompt_token_count', 0) or 0
    _usage['output'] += getattr(meta, 'candidates_token_count', 0) or 0
    _usage['cached'] += getattr(meta, 'cached_content_token_count', 0) or 0
    _usage['total']  += getattr(meta, 'total_token_count', 0) or 0


def usage_cost() -> float:
    return (_usage['input'] / 1e6) * COST_PER_M_INPUT + (_usage['output'] / 1e6) * COST_PER_M_OUTPUT


class BudgetExceeded(RuntimeError):
    """A run budget is spent. Raised before the request is issued, and caught by the
    Run Classification cell, which breaks the loop and saves the rows that completed
    rather than discarding the run."""


class FatalRequestError(BudgetExceeded):
    """A permanent request failure (bad model, bad key, malformed request). Subclasses
    BudgetExceeded so the run loop stops cleanly and saves completed rows."""


class QuotaExhausted(BudgetExceeded):
    """A 429 RESOURCE_EXHAUSTED from the API. Subclasses BudgetExceeded so the run
    loop's existing clean-stop path applies: the run ends immediately and whatever
    completed is still saved and basketed."""


def _is_quota_error(exc: Exception) -> bool:
    text = f'{type(exc).__name__}: {exc}'
    return '429' in text or 'RESOURCE_EXHAUSTED' in text


# Permanent request-level failures. Retrying these cannot help: the model is gone, the
# key is wrong, or the request shape is invalid — the next 299 attempts fail identically.
# Retrying a 404 across 100 rows is 300 pointless requests and ~10 wasted minutes.
_FATAL_MARKERS = ('404', 'NOT_FOUND', 'INVALID_ARGUMENT', 'PERMISSION_DENIED',
                  'UNAUTHENTICATED', '401', '403')


def _is_fatal_error(exc: Exception) -> bool:
    text = f'{type(exc).__name__}: {exc}'
    return any(m in text for m in _FATAL_MARKERS)


_last_request_at = 0.0
_quota_errors = 0


def _throttle() -> None:
    """Sleep just enough to keep requests at most one per REQUEST_INTERVAL_S."""
    global _last_request_at
    if REQUEST_INTERVAL_S > 0:
        wait = REQUEST_INTERVAL_S - (time.time() - _last_request_at)
        if wait > 0:
            time.sleep(wait)
    _last_request_at = time.time()


def _spend_llm_budget() -> None:
    """Check both ceilings before issuing a request. The token ceiling can only be
    tested between calls — usage_metadata arrives with the response — so the guarantee
    is 'no further calls once crossed', with overshoot bounded by a single call."""
    global _llm_calls_made
    # MAX_LLM_TWEETS caps TWEETS; multi-pass labelling issues one call per tweet per pass,
    # so the call ceiling is the product. Stated here rather than letting N passes
    # silently consume N times the budget the constant appears to promise.
    _call_ceiling = MAX_LLM_TWEETS * max(N_LABEL_PASSES, 1)
    if _llm_calls_made >= _call_ceiling:
        raise BudgetExceeded(
            f'Call ceiling reached: {_call_ceiling:,} = MAX_LLM_TWEETS ({MAX_LLM_TWEETS:,}) '
            f'x N_LABEL_PASSES ({N_LABEL_PASSES}). Raise either in the Configuration cell '
            'if that is intended.'
        )
    if _usage['total'] >= MAX_SESSION_TOKENS:
        raise BudgetExceeded(
            f'MAX_SESSION_TOKENS ({MAX_SESSION_TOKENS:,}) reached — '
            f'{_usage["total"]:,} tokens spent over {_usage["calls"]:,} calls. '
            'Raise MAX_SESSION_TOKENS in the Configuration cell if that is intended.'
        )
    _llm_calls_made += 1

def _strip_code_fences(raw: str) -> str:
    raw = raw.strip()
    if raw.startswith('```'):
        raw = re.sub(r'^```(?:json)?\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
    return raw.strip()

def classify_tweet(text: str, client_override=None, config_override=None,
                   model_override=None) -> dict:
    """Label one tweet. The overrides let a multi-pass run use a fresh client, a per-pass
    temperature and a per-pass MODEL without rebuilding this function."""
    _spend_llm_budget()
    _client = client_override if client_override is not None else client
    _config = config_override if config_override is not None else GEN_CONFIG
    _model  = model_override  if model_override  is not None else MODEL_NAME
    prompt = build_prompt(text)
    last_error = ''
    for attempt in range(MAX_RETRIES):
        try:
            _throttle()
            resp = _client.models.generate_content(
                model=_model,
                contents=prompt,
                config=_config,
            )
            _record_usage(resp)
            raw  = _strip_code_fences(resp.text or '')
            parsed = json.loads(raw)
            if not isinstance(parsed, list):
                raise ValueError(f"Expected array, got {type(parsed)}")
            results = []
            for p in parsed:
                cat = str(p.get('category', '')).strip()
                if cat not in CATEGORIES:
                    raise ValueError(f'category {cat!r} not in CATEGORIES')
                results.append({
                    'category': cat,
                    'confidence': float(p.get('confidence', 0.0)),
                    'rationale': str(p.get('rationale', ''))[:500],
                })
            return results
        except Exception as e:
            last_error = f'{type(e).__name__}: {e}'
            if _is_fatal_error(e):
                raise FatalRequestError(
                    f'Permanent request failure on call {_llm_calls_made:,} — retrying cannot '
                    f'fix it, so the run stops here and keeps whatever completed. '
                    f'Original: {last_error[:300]}'
                ) from e
            if _is_quota_error(e):
                # A 429 is either transient throttling (ride it out with a long wait) or
                # genuine exhaustion (retrying only spends more of what is gone). Give it
                # the long backoffs, then abort — never the fast retry loop below, which
                # is what burned a whole free-tier quota in one run.
                global _quota_errors
                _quota_errors += 1
                if _quota_errors <= len(QUOTA_BACKOFF_S) and _quota_errors <= MAX_QUOTA_ERRORS:
                    _wait = QUOTA_BACKOFF_S[_quota_errors - 1]
                    print(f'  429 #{_quota_errors} — waiting {_wait}s for the rate window to reset')
                    time.sleep(_wait)
                    continue
                raise QuotaExhausted(
                    f'429 RESOURCE_EXHAUSTED after {_llm_calls_made:,} calls and '
                    f'{_quota_errors} quota errors this run. Backoffs {QUOTA_BACKOFF_S} did not '
                    f'clear it, so this is exhaustion rather than throttling. Check '
                    f'https://ai.dev/rate-limit or the project quota page. '
                    f'Original: {last_error[:200]}'
                ) from e
            global _first_error_reported
            if not _first_error_reported:
                # Surface the FIRST failure immediately. Without this, a request-shape
                # error is invisible until the whole run finishes — three silent retries
                # per row turn a 20-second diagnosis into a 10-minute one.
                _first_error_reported = True
                print(f'\n*** FIRST ERROR (retrying {MAX_RETRIES}x per row): {last_error[:400]} ***\n')
            if attempt + 1 < MAX_RETRIES:
                time.sleep(INITIAL_BACKOFF * (2 ** attempt))
    return [{'category': 'PARSE_ERROR', 'confidence': 0.0, 'rationale': last_error[:500]}]

# 2 · Preflight

**Run this before the labelling loop.** One request with the real prompt and real config,
printing either the parsed response or the full exception.

A malformed request otherwise stays invisible until the whole run ends: at three retries per
row, 100 rejected tweets take ~10 minutes to report what one call reports in two seconds.

## Connection Check — One Call

In [ ]:
%%time
# The Run section refuses to start unless this cell has set PREFLIGHT_OK = True.
PREFLIGHT_OK = False
_probe_text = 'ai art is just theft, it copies real artists and calls it new'

# EVERY distinct model is preflighted, not just the first. A model that is only reached
# in pass 2 would otherwise fail for the first time after pass 1 has already been paid
# for — and a 404 or a rejected config fails identically on every subsequent row.
try:
    for _pm in ACTIVE_MODELS:
        print(f'--- {_pm} ---')
        _r = client.models.generate_content(
            model=_pm, contents=build_prompt(_probe_text),
            config=types.GenerateContentConfig(**MODEL_KWARGS[_pm]),
        )
        print('REQUEST OK')
        print('  raw response :', (_r.text or '')[:300])
        _m = getattr(_r, 'usage_metadata', None)
        if _m is not None:
            print(f'  tokens       : input={_m.prompt_token_count}, '
                  f'output={_m.candidates_token_count}, '
                  f'cached={getattr(_m, "cached_content_token_count", 0)}')
        else:
            print('  tokens       : usage_metadata absent on this response')
        _parsed = json.loads(_strip_code_fences(_r.text or ''))
        for p in _parsed:
            assert p['category'] in CATEGORIES, f'category {p["category"]!r} not in CATEGORIES'
        print(f'  parsed       : {_parsed}')
    PREFLIGHT_OK = True
    print(f'\nPreflight passed for all {len(ACTIVE_MODELS)} model(s) — safe to run the loop.')
except Exception as _e:
    print('REQUEST FAILED — do NOT run the labelling loop yet.')
    print(f'  {type(_e).__name__}: {_e}')
    print('\nCommon causes:')
    print('  * INVALID_ARGUMENT naming response_json_schema / response_schema')
    print('       -> set USE_RESPONSE_SCHEMA = False in Configuration, re-run the client cell.')
    print('  * NOT_FOUND \'no longer available to new users\'')
    print('       -> that model is closed to new accounts. Re-run the Available Models cell')
    print('          and pick a listed model from a NEWER family (the whole gemini-2.5 line')
    print('          was closed to new users on 2026-08-03), then re-run Configuration.')
    print('  * 401 / API key not valid')
    print('       -> the Colab secret holds a Vertex-AI key; mint an AI Studio key instead.')
    raise

# 3 · Load and Estimate

Load the partition, apply the deterministic nested selection and the row cap, then project
token cost **before** any spend.

## Load Input

In [ ]:
%%time
assert INPUT_PATH.exists(), f'Input not found: {INPUT_PATH}. Run 00_hitl_data_preparation.ipynb first.'
df = pd.read_pickle(INPUT_PATH)
print(f'Loaded {len(df):,} tweets from {INPUT_PATH.name}')

for col in ('id', 'text', 'likes', 'retweets'):
    if col not in df.columns:
        df[col] = '' if col in ('id', 'text') else 0

df['text'] = df['text'].astype(str)
df['id'] = df['id'].astype(str)

# Deterministic, NESTED selection. Sort by id first so the permutation depends only on
# WHICH tweets are in the partition, not on the order the pickle happens to store them
# in; then take a head slice. head(100) is therefore a strict subset of head(1000), so a
# smoke run and a later full run label overlapping tweets and the basket below grows
# monotonically. (Two `df.sample(n=...)` calls sharing a seed but differing in n do NOT
# nest — they would have produced two largely disjoint sets.)
df = df.sort_values('id').sample(frac=1, random_state=SELECTION_SEED).reset_index(drop=True)

n_target = min(SMOKE_TEST_N if SMOKE_TEST else len(df), MAX_LLM_TWEETS)
df = df.head(n_target).reset_index(drop=True)

assert len(df) <= MAX_LLM_TWEETS, f'{len(df):,} rows exceeds MAX_LLM_TWEETS ({MAX_LLM_TWEETS:,})'
print(f'Selection: {"SMOKE_TEST" if SMOKE_TEST else "full"} → first {len(df):,} of the fixed '
      f'permutation (seed {SELECTION_SEED}); MAX_LLM_TWEETS={MAX_LLM_TWEETS:,}')
print(f'Will send {len(df):,} tweets to {MODEL_NAME}')

## Token and Cost Estimate

`TOKENOPT_REF.md` §17. Built from the real prompts for the rows actually loaded, so it tracks
the current criteria. Output is an upper bound (assumes every response fills
`MAX_OUTPUT_TOKENS`); the measured figures after the run are authoritative.

In [ ]:
%%time
_prompts     = [build_prompt(t) for t in df['text']]
_chars       = sum(len(p) for p in _prompts)
_static      = len(build_prompt(''))
est_input    = _chars / CHARS_PER_TOKEN
est_output   = len(df) * MAX_OUTPUT_TOKENS
_passes      = max(N_LABEL_PASSES, 1)
est_cost     = (est_input / 1e6 * COST_PER_M_INPUT
                + est_output / 1e6 * COST_PER_M_OUTPUT) * max(N_LABEL_PASSES, 1)

print(f'{len(df):,} tweets x {_passes} pass(es) = {len(df) * _passes:,} calls; '
      f'pass models {PASS_MODELS_RESOLVED}')
print(f'  est. input  : {est_input:>12,.0f} tokens')
print(f'  est. output : {est_output:>12,.0f} tokens  (upper bound: MAX_OUTPUT_TOKENS each)')
print(f'  est. cost   : ${est_cost:>11,.4f}  @ ${COST_PER_M_INPUT}/M in, '
      f'${COST_PER_M_OUTPUT}/M out'
      + ('  (dearest model\'s rates — UPPER BOUND)' if RATES_ARE_WORST_CASE else ''))
print(f'  static scaffold: {_static:,} of {_chars/max(len(df),1):,.0f} chars per prompt '
      f'({_static/(_chars/max(len(df),1)):.0%}) — identical every call, so implicit-cache eligible')

if est_input + est_output > MAX_SESSION_TOKENS:
    _per_call = (est_input + est_output) / max(len(df), 1)
    print(f'\n*** MAX_SESSION_TOKENS ({MAX_SESSION_TOKENS:,}) is BELOW this run\'s estimated '
          f'{est_input + est_output:,.0f} tokens.')
    print(f'    Expect the run to stop after roughly {int(MAX_SESSION_TOKENS / _per_call):,} '
          f'of {len(df):,} tweets. Completed rows are saved and basketed as normal. ***')

if est_cost > COST_ALERT_USD:
    print(f'\n*** COST ALERT: ${est_cost:,.2f} exceeds COST_ALERT_USD (${COST_ALERT_USD:,.2f}). '
          f'Review before running the next cell. ***')

del _prompts   # a full copy of every prompt; no reason to hold it through the run

# 4 · Run

The labelling loop. Checkpoints every `CHECKPOINT_EVERY` rows, stops cleanly on either spend
guard, and reports measured token usage against the estimate.

## Labelling Loop

In [ ]:
%%time
# Preflight gate. A failed preflight means every call in the loop fails the same way;
# without this the loop happily issues MAX_RETRIES x len(df) doomed requests.
if not globals().get('PREFLIGHT_OK', False):
    raise RuntimeError(
        'Preflight has not passed. Run the Preflight cell (section 2) and fix what it '
        'reports before starting the loop. Override only if you know why: PREFLIGHT_OK = True.')

assert len(df) <= MAX_LLM_TWEETS, f'{len(df):,} rows exceeds MAX_LLM_TWEETS ({MAX_LLM_TWEETS:,})'
_llm_calls_made = 0   # reset the per-run budget so this cell is safely re-runnable
_reset_usage()        # and the token accumulator, for the same reason
_quota_errors = 0     # and the 429 counter

_temps = PASS_TEMPERATURES or [TEMPERATURE] * N_LABEL_PASSES
assert len(_temps) >= N_LABEL_PASSES, (
    f'PASS_TEMPERATURES has {len(_temps)} entries but N_LABEL_PASSES is {N_LABEL_PASSES}')

print(f'{N_LABEL_PASSES} pass(es) over {len(df):,} tweets '
      f'= {N_LABEL_PASSES * len(df):,} requests')
for _p in range(N_LABEL_PASSES):
    print(f'  pass {_p + 1}: {PASS_MODELS_RESOLVED[_p]} @ T={_temps[_p]}')

t0 = time.time()
stopped_early = False
passes: list[dict] = []          # pass index -> {tweet id: classification dict}

for _p in range(N_LABEL_PASSES):
    # A fresh client per pass, so passes share no client-side state. This does NOT
    # guarantee the provider treats them as unrelated requests — it removes our end of
    # the coupling, not theirs.
    _pass_client = genai.Client(api_key=API_KEY)
    # That pass's model config (thinking budget as that model accepted it), with this
    # pass's temperature layered on top.
    _pass_model  = PASS_MODELS_RESOLVED[_p]
    _pass_config = types.GenerateContentConfig(
        **{**MODEL_KWARGS[_pass_model], 'temperature': _temps[_p]})
    _pass_out: dict = {}

    for _, row in tqdm.tqdm(df.iterrows(), total=len(df),
                            desc=f'pass {_p + 1}/{N_LABEL_PASSES} '
                                 f'({_pass_model} T={_temps[_p]})'):
        try:
            _pass_out[row['id']] = classify_tweet(
                row['text'], client_override=_pass_client, config_override=_pass_config,
                model_override=_pass_model)
        except BudgetExceeded as e:
            stopped_early = True
            tqdm.tqdm.write(f'\n*** STOPPED in pass {_p + 1}: {e}')
            tqdm.tqdm.write(f'    {len(_pass_out):,} of {len(df):,} done in this pass. ***')
            break
    passes.append(_pass_out)
    if stopped_early:
        break

# Every tweet the LLM actually READ, in any pass. This is deliberately computed before
# the completeness filter below: a run that stops mid-pass-2 leaves tweets that pass 1
# labelled and pass 2 never reached, and those get dropped from out_df. The model still
# read them, so they are still training data — basketing only the surviving rows would
# quietly release them back into the pool that held-out evaluation draws from, which is
# the exact leak the basket exists to prevent.
sent_ids = sorted({i for p in passes for i in p})

# Keep only tweets that every pass managed to label, so agreement is computed on a
# complete matrix rather than a ragged one.
_complete = [i for i in df['id'] if all(i in p for p in passes)]
if len(_complete) < len(df):
    print(f'\n{len(df) - len(_complete):,} tweet(s) missing from at least one pass — excluded '
          f'from out_df, but still basketed as seen ({len(sent_ids):,} sent in total).')

results = []
for _, row in df[df['id'].isin(_complete)].iterrows():
    _labels = [sorted([x['category'] for x in passes[p][row['id']]]) for p in range(len(passes))]
    _agree  = all(tuple(l) == tuple(_labels[0]) for l in _labels)
    
    # Pass 1 is operative
    operative_preds = passes[0][row['id']]
    op_cats = [x['category'] for x in operative_preds]
    
    rec = {
        'id': row['id'],
        'text': row['text'],
        'likes': row.get('likes', 0),
        'retweets': row.get('retweets', 0),
        'passes_agree': _agree,
    }
    
    # one-hot pred and human label columns
    for cat in CATEGORIES:
        if cat == 'none': continue
        rec[f'pred_{cat}'] = 1 if cat in op_cats else 0
        rec[f'human_{cat}'] = ''
    rec['pred_none'] = 1 if 'none' in op_cats else 0
    rec['human_none'] = ''
        
    for p in range(len(passes)):
        rec[f'predictions_pass{p + 1}'] = passes[p][row['id']]
    rec['parse_error'] = 1 if sum([rec.get(f'pred_{c}', 0) for c in CATEGORIES]) == 0 else 0
    results.append(rec)

out_df = pd.DataFrame(results)
print(f'\n{"STOPPED EARLY" if stopped_early else "Done"}. '
      f'{len(out_df):,} of {len(df):,} tweets labelled by {len(passes)} pass(es). '
      f'Total time: {time.time()-t0:.1f}s')
print('Prediction sums:')
for cat in CATEGORIES:
    if f'pred_{cat}' in out_df:
        print(f'{cat}: {out_df[f"pred_{cat}"].sum()}')

if len(passes) > 1 and len(out_df):
    _n_dis = int((~out_df['passes_agree']).sum())
    print(f"\nInter-pass agreement: {1 - _n_dis / len(out_df):.1%} "
          f"({_n_dis:,} disagreement(s) of {len(out_df):,}) — see the Review section.")

# TOKENOPT_REF.md §16 — measured, not estimated.
if _usage['calls']:
    _expected = len(out_df) * len(passes)
    print(f"\nMeasured usage over {_usage['calls']:,} API calls "
          f"({max(_usage['calls'] - _expected, 0):,} of them retries):")
    print(f"  input   : {_usage['input']:>12,} tokens")
    print(f"  cached  : {_usage['cached']:>12,} tokens "
          f"({_usage['cached']/max(_usage['input'],1):.1%} of input — implicit cache hit rate)")
    print(f"  output  : {_usage['output']:>12,} tokens")
    print(f"  total   : {_usage['total']:>12,} tokens")
    _est = globals().get('est_cost')
    _cmp = (f'  (est. was ${_est * len(passes):,.4f} for {len(passes)} passes)'
            if _est is not None else '  (no pre-flight estimate)')
    print(f"  cost    : ${usage_cost():,.4f}{_cmp}")
else:
    print('\nNo usage_metadata captured — the SDK may not expose it on this response type.')


# 5 · Review

Eyeball the labelling before it becomes training data. Distribution, confidence, a cue-word
audit, and worked examples — enough to judge whether `categories.md` is doing what you meant.

Nothing here calls the API or writes to disk, so it is free to re-run.

## Classification Report

In [ ]:
%%time
# ─── Knobs for this report ───────────────────────────────────────────────
LOW_CONF_BELOW = 0.8
N_POSITIVE = 100
N_SAMPLES  = 10
# ────────────────────────────────────────────────────────────────────────

import textwrap

# Extract cue words logic
CUE_WORDS = [
    'steal', 'stole', 'stolen', 'theft', 'thief', 'plagiaris', 'copy',
    'scrap', 'consent', 'permission', 'compensat', 'credit', 'dataset',
    'real art', 'human art', 'soul', 'effort', 'talent', 'lazy',
    'luddite', 'gatekeep', 'adapt', 'tool', 'prompt', 'democratiz',
]

def _has_cue(text: str) -> bool:
    low = str(text).lower()
    return any(w in low for w in CUE_WORDS)

def _show(rows, title: str) -> None:
    print(f'\n{title}')
    print('-' * 80)
    if len(rows) == 0:
        print('  (none)')
        return
    for _, r in rows.iterrows():
        cue = 'cue' if _has_cue(r['text']) else 'no-cue'
        # Get active categories
        active_cats = [c for c in CATEGORIES if r.get(f'pred_{c}', 0) == 1]
        print(f"  [{', '.join(active_cats)}]  conf={r.get('confidence',0):.2f}  {cue}"
              f"  likes={r.get('likes', 0)}  rt={r.get('retweets', 0)}")
        for line in textwrap.wrap(str(r['text']), 80 - 6) or ['']:
            print(f'      {line}')
        for line in textwrap.wrap(f"why: {r.get('rationale','')}", 80 - 6):
            print(f'      | {line}')
        print()

out_df['_cue'] = out_df['text'].map(_has_cue)

# We added 'parse_error' in out_df or we can deduce it if no category is 1.
out_df['parse_error'] = (out_df[[f'pred_{c}' for c in CATEGORIES]].sum(axis=1) == 0).astype(int)
ok = out_df[out_df['parse_error'] == 0].copy()
if 'confidence' not in ok:
    ok['confidence'] = 0.0
ok['confidence'] = pd.to_numeric(ok['confidence'], errors='coerce').fillna(0.0)

print('=' * 80)
print(f'CLASSIFICATION REPORT — {len(out_df):,} rows')
print('=' * 80)

# ─── 1. Label distribution ───────────────────────────────────────────────
print('\nLABEL DISTRIBUTION (Multi-label)')
print('-' * 80)
for cat in CATEGORIES:
    n = int(ok[f'pred_{cat}'].sum()) if f'pred_{cat}' in ok else 0
    bar = '#' * int(40 * n / max(len(out_df), 1))
    print(f'  {cat:<20} {n:>6,}  {n/len(out_df):>6.1%}  {bar}')

n_err = int(out_df['parse_error'].sum())
if n_err:
    print(f'\n  WARNING: {n_err:,} PARSE_ERROR rows are excluded from everything below.')

# ─── 2. Confidence ───────────────────────────────────────────────────────
print('\nCONFIDENCE BY LABEL')
print('-' * 80)
print(f"  {'label':<20} {'n':>6} {'mean':>7} {'median':>7} {'min':>6} {'< ' + str(LOW_CONF_BELOW):>9}")
for cat in CATEGORIES:
    if f'pred_{cat}' in ok:
        grp = ok[ok[f'pred_{cat}'] == 1]
        if len(grp) > 0:
            print(f"  {cat:<20} {len(grp):>6} {grp['confidence'].mean():>7.2f} "
                  f"{grp['confidence'].median():>7.2f} {grp['confidence'].min():>6.2f} "
                  f"{(grp['confidence'] < LOW_CONF_BELOW).sum():>9}")

# ─── 3. Samples ──────────────────────────────────────────────────────────
for label in CATEGORIES:
    if f'pred_{label}' in ok:
        grp = ok[ok[f'pred_{label}'] == 1]
        _show(grp.sample(n=min(N_SAMPLES, len(grp)), random_state=42),
              f'RANDOM SAMPLE — {label} ({len(grp):,} rows)')


# 6 · Save

Four artifacts: the HITL-schema CSV, the full pickle with `confidence` and `rationale`, two
JSON records — measured token usage and the accumulating basket of tweet ids the LLM has
seen — and a verbatim copy of the prompt, byte-identical to the repo's.
The CSV is ordered positive-class-first, and carries the model's `rationale` so a
reviewer can see why each label was chosen.
**The basket is training data: exclude those ids from any held-out evaluation.**

## Outputs, Usage Record, and Seen-Ids Basket

In [ ]:
%%time
import hashlib

_label_rank = {_lab: _i for _i, _lab in enumerate(CATEGORIES)}
out_df = out_df.sort_values(
    'confidence',
    ascending=False,
    kind='stable',
).reset_index(drop=True)

# Flatten rationale
for _rcol in [c for c in out_df.columns if c == 'rationale' or c.startswith('rationale_pass')]:
    out_df[_rcol] = (out_df[_rcol].astype(str)
                     .str.replace(r'\s+', ' ', regex=True).str.strip())

_disagree_cols = []
if 'passes_agree' in out_df.columns:
    _disagree_cols = ['passes_agree']

_pass_models = list(PASS_MODELS_RESOLVED[:len(passes)]) if 'passes' in globals() else [MODEL_NAME]
_provenance_cols = []
for _p in range(len(_pass_models)):
    out_df[f'model_pass{_p + 1}'] = _pass_models[_p]
    out_df[f'temp_pass{_p + 1}'] = _temps[_p] if '_temps' in globals() else TEMPERATURE
    _provenance_cols += [f'model_pass{_p + 1}', f'temp_pass{_p + 1}']

human_cols = [f'human_{c}' for c in CATEGORIES]
pred_cols = [f'pred_{c}' for c in CATEGORIES]

hitl_schema_cols = (['id', 'text', 'likes', 'retweets'] + pred_cols + ['rationale']
                    + _disagree_cols + human_cols + _provenance_cols)
hitl_schema_cols = [c for c in hitl_schema_cols if c in out_df.columns]

out_df[hitl_schema_cols].to_csv(OUTPUT_CSV, index=False)
out_df.to_pickle(OUTPUT_PKL)

n_errors = int(out_df['parse_error'].sum())
print(f'Saved → {OUTPUT_CSV}')
print(f'Saved → {OUTPUT_PKL}')
print(f'PARSE_ERROR rows: {n_errors:,} / {len(out_df):,} ({n_errors/max(len(out_df),1):.1%})')

usage_record = {
    'timestamp':       time.strftime('%Y-%m-%dT%H:%M:%S'),
    'model':           MODEL_NAME,
    'dataset_type':    DATASET_TYPE,
    'rows':            int(len(out_df)),
    'prompt_chars':    len(PROMPT_TEMPLATE),
    'prompt_doc_sha256':  PROMPT_DOC_SHA256,
    'prompt_sent_sha256': hashlib.sha256(PROMPT_TEMPLATE.encode('utf-8')).hexdigest()[:12],
    'parse_errors':    int(n_errors),
}
USAGE_JSON = hitl_folder / f'llm_bootstrap_usage_{usage_record["timestamp"].replace(":", "")}.json'
with open(USAGE_JSON, 'w') as f:
    json.dump(usage_record, f, indent=2)
print(f'Saved → {USAGE_JSON}')

SEEN_IDS_JSON = hitl_folder / f'llm_bootstrap_seen_ids_{DATASET_TYPE}.json'
seen = {
    'note': 'Tweet ids sent to the LLM for bootstrap labelling. Accumulates across runs.',
    'dataset_type': DATASET_TYPE,
    'ids': [],
    'parse_error_ids': [],
    'runs': [],
}
if SEEN_IDS_JSON.exists():
    with open(SEEN_IDS_JSON) as f:
        seen = json.load(f)

run_ids = [str(i) for i in globals().get('sent_ids', out_df['id'])]
err_ids = {str(i) for i in out_df.loc[out_df['parse_error'] == 1, 'id']}
ok_ids  = {str(i) for i in out_df.loc[out_df['parse_error'] == 0, 'id']}

seen['ids'] = sorted(set(seen['ids']) | set(run_ids))
_old_errs = set(seen['parse_error_ids'])
seen['parse_error_ids'] = sorted((_old_errs | err_ids) - ok_ids)
seen['runs'].append(usage_record)

with open(SEEN_IDS_JSON, 'w') as f:
    json.dump(seen, f, indent=2)
print(f'Basket updated → {SEEN_IDS_JSON} (total {len(seen["ids"]):,} seen)')


## Prompt Record for Reviewers

Writes `llm_bootstrap_prompt.md` next to the CSV — the same file that lives in the repo, byte
for byte — so whoever fills in `human_label` is reading exactly the text the model was given.

Nothing run-specific is added to it. A generated header would make the Drive copy differ from
the repo copy, and then neither could be trusted as *the* prompt. The run's model, passes,
temperatures and token spend go into the usage record instead, keyed by the same `sha256`.

In [ ]:
%%time
# PROMPT_DOC is llm_bootstrap_prompt.md, embedded verbatim by sync_prompt.py. Writing it
# out unchanged is what makes the reviewer's copy and the author's copy the same artifact:
# a reviewer reading it in Drive and an author editing it in git see identical bytes, and
# the assertion below is what turns that from an intention into a checked property.
PROMPT_MD_OUT = hitl_folder / 'llm_bootstrap_prompt.md'
PROMPT_MD_OUT.write_text(PROMPT_DOC, encoding='utf-8')
assert PROMPT_MD_OUT.read_text(encoding='utf-8') == PROMPT_DOC, 'prompt record did not round-trip'

print(f'Saved → {PROMPT_MD_OUT}')
print(f'  {len(PROMPT_DOC):,} chars, sha256:{PROMPT_DOC_SHA256} — byte-identical to '
      f'the repo\'s notebooks/05_Classifiers/llm_bootstrap_prompt.md')

# 7 · Teardown

Releases the Colab runtime when `AUTO_DISCONNECT` is True. Keep it **False** while iterating —
unassigning destroys every variable, so a failed run leaves nothing to inspect.

## Disconnect

In [ ]:
# Disconnect from Colab runtime (no-op locally).
# Gated by AUTO_DISCONNECT: unassigning frees the runtime but DESTROYS all variables,
# so a failed run leaves nothing to inspect. Keep it False while iterating; set it True
# for a long unattended full run where releasing the instance matters.
if AUTO_DISCONNECT:
    try:
        from google.colab import runtime
        runtime.unassign()
    except ImportError:
        pass
else:
    print('AUTO_DISCONNECT is False — runtime left running so out_df stays inspectable.')
